In [ ]:

import sys
from pathlib import Path

# Make dissector importable without a formal install
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dissector.evaluation import radarplot_single_dataset

DATA_DIR = Path('final_results_copy')


In [ ]:
# ── Load summary CSVs and extract MuscleMap rows ──────────────────────────────
#
# Each dataset contributes one row per algorithm (already averaged over subjects).
# We collect those rows so each panel gets one spider line per dataset.

import re

SEQ_PATTERN = re.compile(
    r'\s*\((water only|water|fat fraction|fat_fraction|dixon|Dixon|both channels?|both)\)\s*$',
    re.IGNORECASE
)

DATASETS = {
    'MyoSegmenTUM': 'overall_means_myosegmentum.csv',
    'Pathological': 'overall_means_P_only.csv',
    'AIPS':         'overall_means_asian.csv',
    'Sheffield':    'overall_means_sheffield.csv',
    'Augmented':    'overall_means_augmented.csv',
}

DATASET_COLORS = {
    'MyoSegmenTUM': '#1f77b4',
    'Pathological': '#ff7f0e',
    'AIPS':         '#2ca02c',
    'Sheffield':    '#d62728',
    'Augmented':    '#9467bd',
}

# Metrics for the radar — all must be in [0, 1], higher = better.
# hausdorff is normalised: hd_score = 1 - clip(HD / HD_MAX, 0, 1)
HD_MAX = 500  # mm — anything at or above this scores 0

RADAR_COLS  = ['dice', 'jaccard', 'boundary_iou_3d', 'recall', 'hd_score']
RADAR_LABELS = ['Dice', 'Jaccard\n(IoU)', 'Boundary\nIoU', 'Recall\n(1\u2212FN)', 'HD\nscore']


def build_radar_df(algorithm_base: str) -> pd.DataFrame:
    """Return a DataFrame with one row per dataset, columns = RADAR_COLS."""
    rows = []
    for ds_name, fname in DATASETS.items():
        df = pd.read_csv(DATA_DIR / fname)
        df['base'] = df['algorithm'].apply(lambda n: SEQ_PATTERN.sub('', n).strip())
        subset = df[df['base'] == algorithm_base]
        if subset.empty:
            continue
        m = subset[['dice', 'jaccard', 'boundary_iou_3d', 'false_negative', 'hausdorff']].mean()
        rows.append({
            'dataset':        ds_name,
            'dice':           float(m['dice']),
            'jaccard':        float(m['jaccard']),
            'boundary_iou_3d':float(m['boundary_iou_3d']),
            'recall':         1.0 - float(m['false_negative']),
            'hd_score':       1.0 - min(float(m['hausdorff']) / HD_MAX, 1.0),
        })
    return pd.DataFrame(rows).set_index('dataset')


df_wb    = build_radar_df('MuscleMap WB')
df_thigh = build_radar_df('MuscleMap Thigh')

print('MuscleMap WB:\n',    df_wb.round(3))
print('\nMuscleMap Thigh:\n', df_thigh.round(3))

In [ ]:
# ── Quick preview using radarplot_single_dataset from dissector.evaluation ────
# (Each row in the DataFrame becomes one spider line; all are plotted in blue.)

radarplot_single_dataset(
    metrics=RADAR_COLS,
    dataset=df_wb.reset_index(),
    title='MuscleMap WB — per-dataset profiles',
    labels=RADAR_LABELS,
)

radarplot_single_dataset(
    metrics=RADAR_COLS,
    dataset=df_thigh.reset_index(),
    title='MuscleMap Thigh — per-dataset profiles',
    labels=RADAR_LABELS,
)

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# Side-by-side coloured radar — MuscleMap WB (left) vs MuscleMap Thigh (right)
#
# Uses the same polar-plot pattern as radarplot_single_dataset but:
#   • each dataset gets its own colour
#   • both panels share a legend
#   • a filled polygon shows the mean profile
# ═══════════════════════════════════════════════════════════════════════════════

PLOT_COLS   = ['dice', 'jaccard', 'boundary_iou_3d', 'recall']
PLOT_LABELS = ['Dice', 'Jaccard\n(IoU)', 'Boundary\nIoU', 'Recall\n(1−FN)']

num_metrics = len(PLOT_COLS)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False)
angles = np.append(angles, angles[0])  # close the loop

fig, axes = plt.subplots(1, 2, figsize=(13, 6), subplot_kw={'polar': True})
fig.subplots_adjust(wspace=0.45)

panels = [
    (axes[0], df_wb,    'MuscleMap WB (v2.0)'),
    (axes[1], df_thigh, 'MuscleMap Thigh (v2.0)'),
]

for ax, df, title in panels:
    # Individual dataset lines
    for ds_name, row in df.iterrows():
        vals = row[PLOT_COLS].to_numpy(dtype=float)
        vals = np.append(vals, vals[0])
        color = DATASET_COLORS.get(ds_name, '#888888')
        ax.plot(angles, vals, 'o-', linewidth=1.8, color=color, alpha=0.75, markersize=4)
        ax.fill(angles, vals, alpha=0.00, color=color)

    # Mean profile (thick dashed, semi-transparent fill)
    mean_vals = df[PLOT_COLS].mean().to_numpy(dtype=float)
    mean_vals = np.append(mean_vals, mean_vals[0])
    ax.plot(angles, mean_vals, linewidth=2.5, color='black', linestyle='--',
            alpha=0.6, label='Mean', zorder=5)
    ax.fill(angles, mean_vals, alpha=0.00, color='black')

    # Axes formatting
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(PLOT_LABELS, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.0'], fontsize=7, color='grey')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=18)
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

# Shared legend
handles = [
    mpatches.Patch(color=DATASET_COLORS[ds], label=ds)
    for ds in DATASETS if ds in df_wb.index or ds in df_thigh.index
]
handles.append(plt.Line2D([0], [0], color='black', linestyle='--',
                           linewidth=2.5, alpha=0.6, label='Mean'))

fig.legend(handles=handles, title='Dataset', loc='lower center',
           ncol=len(handles), fontsize=9, framealpha=0.9,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('MuscleMap WB vs MuscleMap Thigh — Multi-dataset Metric Profiles',
             fontsize=12, fontweight='bold', y=1.02)

plt.savefig('radar_musclemap_wb_vs_thigh.pdf', bbox_inches='tight')
plt.savefig('radar_musclemap_wb_vs_thigh.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved radar_musclemap_wb_vs_thigh.pdf / .png')
